In [ ]:
%matplotlib widget

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from ipywidgets import FloatSlider, HBox, HTML, Layout
from IPython.display import display

# ============================================================
# CONTINUOUS DERIVATIVE VS BACKWARD DIFFERENCE
# ============================================================

plt.ioff()

CONTENT_WIDTH = '900px'

plt.rcParams.update({'font.size':10.5,'axes.titlesize':12,'axes.labelsize':10.5,'xtick.labelsize':9,'ytick.labelsize':9,'legend.fontsize':8.5})

# ============================================================
# FIXED SIGNAL PARAMETERS
# ============================================================

OMEGA0 = 2.0
T_END = 8.0

# ============================================================
# STYLE
# ============================================================

display(HTML("""
<style>

.ex-root{
    width:900px;
    max-width:900px;
    font-family:Arial,sans-serif;
}

.ex-header{
    background:linear-gradient(90deg,#1565c0,#1976d2);
    color:white;
    padding:11px 15px;
    border-radius:8px 8px 0 0;
    font-size:19px;
    font-weight:bold;
}

.ex-doc{
    background:#f6f9fd;
    border:1px solid #b9cce5;
    border-top:none;
    padding:11px 14px;
    border-radius:0 0 8px 8px;
    font-size:15px;
    line-height:1.55;
    margin-bottom:9px;
}

.ex-box{
    width:100%;
    box-sizing:border-box;
    border:1px solid #b9cce5;
    border-radius:7px;
    padding:10px 12px;
    margin-bottom:8px;
    font-size:14.5px;
    line-height:1.50;
}

.ex-title{
    font-weight:bold;
    color:#0d47a1;
    font-size:15.5px;
    margin-bottom:6px;
}

.ex-cols{
    display:flex;
    gap:18px;
    align-items:flex-start;
    flex-wrap:nowrap;
}

.ex-col{
    flex:1;
    min-width:0;
}

.widget-label{
    font-size:14px !important;
}

.jupyter-widgets input{
    font-size:13.5px !important;
}

.jupyter-widgets-output-area,
.widget-output,
.output_area,
.output_subarea,
.jp-OutputArea-output,
.jp-OutputArea-child{
    overflow-x:visible !important;
    overflow-y:visible !important;
    max-width:none !important;
}

.jp-OutputArea,
.output_wrapper,
.widget-box,
.jupyter-widgets{
    overflow:visible !important;
}

</style>
"""))

# ============================================================
# DOCUMENTATION
# ============================================================

display(HTML("""
<div class="ex-root">

<div class="ex-header">
Continuous Derivative and Backward-Difference Approximation
</div>

<div class="ex-doc">

<b>Continuous derivative.</b>
Consider the continuous-time signal

<br><br>

<div style="text-align:center;font-size:16px;">
<b>x(t) = sin(Ω₀t), &nbsp;&nbsp; Ω₀ = 2 rad/s.</b>
</div>

<br>

Its exact first derivative is

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>x'(t) = Ω₀ cos(Ω₀t).</b>
</div>

When the signal is sampled with sampling period T, the derivative can be approximated by the
first backward difference

<div style="text-align:center;font-size:16px;margin:8px 0;">
<b>
x'[n] ≈ [x[n] - x[n-1]] / T.
</b>
</div>

As the sampling period T decreases, the backward-difference approximation approaches the
continuous derivative evaluated at the sampling instants.

</div>

</div>
"""))

# ============================================================
# CONTROL
# ============================================================

T_slider = FloatSlider(value=0.25,min=0.02,max=0.50,step=0.01,description='Sampling period T:',continuous_update=True,readout_format='.2f',style={'description_width':'125px'},layout=Layout(width='420px'))

design_title = HTML('<div class="ex-title" style="margin:0;">Design parameter</div>',layout=Layout(width='170px'))

controls = HBox([
    design_title,
    T_slider
],layout=Layout(width=CONTENT_WIDTH,border='1px solid #b9cce5',padding='9px 12px',margin='0 0 8px 0',align_items='center'))

info = HTML(layout=Layout(width=CONTENT_WIDTH,margin='0 0 8px 0'))

# ============================================================
# CONTINUOUS SIGNAL
# ============================================================

t_cont = np.linspace(0,T_END,4000)

x_cont = np.sin(OMEGA0*t_cont)
dx_cont = OMEGA0*np.cos(OMEGA0*t_cont)

# ============================================================
# ERROR AS A FUNCTION OF T
# ============================================================

T_values = np.linspace(0.02,0.50,200)
max_errors = np.zeros_like(T_values)

for i,T_test in enumerate(T_values):

    n_test = np.arange(0,int(np.floor(T_END/T_test))+1)
    t_test = n_test*T_test

    x_test = np.sin(OMEGA0*t_test)

    dx_exact_test = OMEGA0*np.cos(OMEGA0*t_test)

    dx_diff_test = np.full_like(x_test,np.nan)

    dx_diff_test[1:] = (x_test[1:]-x_test[:-1])/T_test

    max_errors[i] = np.max(np.abs(dx_exact_test[1:]-dx_diff_test[1:]))

# ============================================================
# FIGURE — CREATED ONCE
# ============================================================

fig,axes = plt.subplots(2,2,figsize=(9.0,6.6))

ax1,ax2,ax3,ax4 = axes.flat

fig.canvas.toolbar_visible = False
fig.canvas.header_visible = False
fig.canvas.footer_visible = False

# ============================================================
# 1. CONTINUOUS SIGNAL AND SAMPLES
# ============================================================

signal_line, = ax1.plot(t_cont,x_cont,color='red',linewidth=1.4,label='Continuous signal')
sample_markers, = ax1.plot([],[],'o',markersize=3.5,label='Samples')

ax1.set_xlim(0,T_END)
ax1.set_ylim(-1.15,1.15)

ax1.set_title('Continuous Signal and Samples')
ax1.set_xlabel('Time $t$')
ax1.set_ylabel('$x(t)$')

ax1.grid(True,linestyle=':',alpha=0.25)
ax1.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 2. EXACT DERIVATIVE AND BACKWARD DIFFERENCE
# ============================================================

derivative_line, = ax2.plot(t_cont,dx_cont,color='red',linewidth=1.4,label='Exact derivative')
difference_line, = ax2.plot([],[],'o-',markersize=3.2,linewidth=1.0,label='Backward difference')

ax2.set_xlim(0,T_END)
ax2.set_ylim(-2.4,2.4)

ax2.set_title('Derivative vs Backward Difference')
ax2.set_xlabel('Time $t$')
ax2.set_ylabel('$dx/dt$')

ax2.grid(True,linestyle=':',alpha=0.25)
ax2.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 3. APPROXIMATION ERROR
# ============================================================

error_line, = ax3.plot([],[],'o-',color='red',markersize=3.2,linewidth=1.0,label='Approximation error')
zero_line = ax3.axhline(0,linestyle='--',linewidth=1.0,label='Zero error')

ax3.set_xlim(0,T_END)
ax3.set_ylim(-1.0,1.0)

ax3.set_title('Backward-Difference Approximation Error')
ax3.set_xlabel('Time $t$')
ax3.set_ylabel('Error')

ax3.grid(True,linestyle=':',alpha=0.25)
ax3.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

# ============================================================
# 4. MAXIMUM ERROR VS SAMPLING PERIOD
# ============================================================

max_error_line, = ax4.plot(T_values,max_errors,color='red',linewidth=1.4,label='Maximum error')
current_T_marker, = ax4.plot([],[],'o',markersize=6,label='Current T')

ax4.set_xlim(0,0.52)
ax4.set_ylim(0,1.05)

ax4.set_title('Maximum Error vs Sampling Period')
ax4.set_xlabel('Sampling period $T$')
ax4.set_ylabel('Maximum absolute error')

ax4.grid(True,linestyle=':',alpha=0.25)
ax4.legend(loc='upper center',bbox_to_anchor=(0.5,-0.17),ncol=2,frameon=False)

plt.subplots_adjust(left=0.08,right=0.98,top=0.94,bottom=0.13,wspace=0.30,hspace=0.62)

# ============================================================
# UPDATE
# ============================================================

def update_derivative(change=None):

    T = T_slider.value

    # --------------------------------------------------------
    # Sampling instants
    # --------------------------------------------------------

    n = np.arange(0,int(np.floor(T_END/T))+1)

    t_samples = n*T

    # --------------------------------------------------------
    # Signal samples
    # --------------------------------------------------------

    x_samples = np.sin(OMEGA0*t_samples)

    # --------------------------------------------------------
    # Exact derivative at the sampling instants
    # --------------------------------------------------------

    exact_derivative_samples = OMEGA0*np.cos(OMEGA0*t_samples)

    # --------------------------------------------------------
    # Backward-difference approximation
    # --------------------------------------------------------

    difference_derivative = np.full_like(x_samples,np.nan)

    difference_derivative[1:] = (x_samples[1:]-x_samples[:-1])/T

    # --------------------------------------------------------
    # Error
    # --------------------------------------------------------

    error = exact_derivative_samples[1:]-difference_derivative[1:]

    max_error = np.max(np.abs(error))
    rms_error = np.sqrt(np.mean(error**2))

    # --------------------------------------------------------
    # Update signal samples
    # --------------------------------------------------------

    sample_markers.set_data(t_samples,x_samples)

    # --------------------------------------------------------
    # Update derivative approximation
    # --------------------------------------------------------

    difference_line.set_data(t_samples[1:],difference_derivative[1:])

    # --------------------------------------------------------
    # Update approximation error
    # --------------------------------------------------------

    error_line.set_data(t_samples[1:],error)

    # --------------------------------------------------------
    # Current point on maximum-error curve
    # --------------------------------------------------------

    current_T_marker.set_data([T],[max_error])

    # --------------------------------------------------------
    # Numerical information
    # --------------------------------------------------------

    fs = 1.0/T
    samples_per_period = (2*np.pi/OMEGA0)/T

    info.value = f"""
    <div class="ex-root">

    <div class="ex-box">

    <div class="ex-title">Current approximation</div>

    <div class="ex-cols">

    <div class="ex-col">
    Signal frequency:<br>
    <b>Ω₀ = {OMEGA0:.2f} rad/s</b><br><br>
    Signal period:<br>
    <b>{2*np.pi/OMEGA0:.4f} s</b>
    </div>

    <div class="ex-col">
    Sampling period:<br>
    <b>T = {T:.2f} s</b><br><br>
    Sampling frequency:<br>
    <b>fₛ = {fs:.2f} Hz</b>
    </div>

    <div class="ex-col">
    Samples per signal period:<br>
    <b>{samples_per_period:.2f}</b><br><br>
    Number of samples:<br>
    <b>{len(t_samples)}</b>
    </div>

    <div class="ex-col">
    Maximum absolute error:<br>
    <b>{max_error:.6f}</b><br><br>
    RMS error:<br>
    <b>{rms_error:.6f}</b>
    </div>

    </div>

    </div>

    </div>
    """

    fig.canvas.draw_idle()

# ============================================================
# EVENT
# ============================================================

T_slider.observe(update_derivative,names='value')

# ============================================================
# DISPLAY
# ============================================================

display(controls)
display(info)
display(fig.canvas)

update_derivative()